In [ ]:
include("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/config/requirements.jl");

We now add noise to the system
\begin{equation}
    \begin{cases}
    \dot p_a = -\beta_a'p_ap_b/\sqrt 2\\
    \dot p_b = 2\biggl[\beta_b'p_a^2/2\sqrt 2  - \varepsilon_d'/\sqrt 2- \kappa_b'p_b \biggr] + \sqrt \epsilon \eta_p(t)\\
    \end{cases}
\end{equation}

The noise is isotropic and we got $$\langle\eta_i(t')\eta_j(t)\rangle = 2\delta_{i,j}\delta(t'-t).$$ We can write MSRDJ action: $$iS=-\int dt [P_{p_a}\dot p_a+P_{p_b}\dot p_b  - H(p_a,p_b,P_{p_a},P_{p_b})],$$
with the Hamiltonian 
\begin{equation}
\begin{split}
H(p_a,p_b,P_{p_a},P_{p_b}) =& [-\beta_a'p_ap_b/\sqrt 2]P_{p_a}+ 2[\beta_b'p_a^2/2\sqrt 2  - \varepsilon_d'/\sqrt 2- \kappa_b'p_b]P_{p_b}+\epsilon P_{p_b}^2
\end{split}
\end{equation}

We parametrize time to solve a geometric problem $$\min_{\lambda\geq0}\min_\phi\max_\theta\int_0^1[\langle\phi'\theta\rangle-\lambda^{-1}H(\phi,\theta)]ds,$$
and we add a constraint on the parametrization of $\phi$ : $|d\phi/ds|=|\phi'|=cst$

We have $\nabla_{\bm\phi}H = \begin{pmatrix}  -\beta_a'p_bP_{p_a}/\sqrt 2+2\beta_b'p_aP_{p_b}/\sqrt 2\\-\beta_a'p_aP_{p_a}/\sqrt 2 - 2\kappa_b'P_{p_b}\end{pmatrix}$ 
and $\nabla_{\bm\theta}H = \begin{pmatrix} -\beta_a'p_ap_b/\sqrt 2\\2\beta_b'p_a^2/2\sqrt 2  - 2\varepsilon_d'/\sqrt 2- 2\kappa_b'p_b + 2\epsilon P_{p_b}\end{pmatrix}$

The fixed points corresponds to 
\begin{equation}
    \begin{cases}
    p_a = 0 \qquad p_b = - \varepsilon_d'/\sqrt 2\kappa_b' \\
    p_a^2 = 2\varepsilon_d'/\beta_b' \qquad p_b = 0 \\
    \end{cases}
\end{equation}

In [ ]:
@inline lerp(a,b,t) = (1-t)*a + t*b

function reparametrize_φ_alone(φ::AbstractMatrix{T},Nout::Int = size(φ, 2);epslen::T = T(1e-14)) where {T<:Real}
    d, N = size(φ)
    @assert N ≥ 2 "Need at least 2 points"

    # --- cumulative arc-length s[i] ---
    s = zeros(T, N)
    @inbounds for i in 2:N
        # Euclidean distance between consecutive points
        di = zero(T)
        @inbounds for k in 1:d
            δ = φ[k,i] - φ[k,i-1]
            di += δ*δ
        end
        di = sqrt(di)
        s[i] = s[i-1] + (di > epslen ? di : zero(T))
    end

    L = s[end]
    # Uniform target arc-length positions
    sr = range(zero(T), L, length = Nout)
    # Output containers
    φr = similar(φ, d, Nout)
    # --- helper: linear interpolation at target s ---
    # For each target sr[j], find segment i-1 -> i s.t. s[i-1] <= sr <= s[i]
    @inbounds for j in 1:Nout
        sj = sr[j]
        i = searchsortedfirst(s, sj)  # i in 1..N
        if i <= 1
            # clamp to first point
            @inbounds for k in 1:d
                φr[k,j] = φ[k,1]
            end
        elseif i > N
            # clamp to last point
            @inbounds for k in 1:d
                φr[k,j] = φ[k,N]
            end
        else
            s0, s1 = s[i-1], s[i]
            t = (s1 - s0) ≤ epslen ? zero(T) : (sj - s0) / (s1 - s0)
            @inbounds for k in 1:d
                φr[k,j] = (one(T)-t)*φ[k,i-1] + t*φ[k,i]
            end
        end
    end

    return φr
end


"""
Reparameterize a discrete curve φ (d×N) to have uniform arc-length spacing.

Inputs
- φ::AbstractMatrix{T}: size (d, N), columns are points φ[:,i] ∈ R^d
- payload::AbstractArray: either
    * vector length N, or
    * matrix size (k, N) (any "data attached to points")
- Nout::Int: number of output points (default = N)

Options
- epslen: threshold to ignore zero-length segments

Returns
- φr: (d, Nout) reparameterized points
- payloadr: payload interpolated to the new parameterization (same "shape", but Nout points)
"""
function reparametrization_arc_length(
    φ::AbstractMatrix{T},
    payload::AbstractArray,
    Nout::Int = size(φ, 2);
    epslen::T = T(1e-14),
) where {T<:Real}

    d, N = size(φ)
    @assert N ≥ 2 "Need at least 2 points"

    # --- cumulative arc-length s[i] ---
    s = zeros(T, N)
    @inbounds for i in 2:N
        # Euclidean distance between consecutive points
        di = zero(T)
        @inbounds for k in 1:d
            δ = φ[k,i] - φ[k,i-1]
            di += δ*δ
        end
        di = sqrt(di)
        s[i] = s[i-1] + (di > epslen ? di : zero(T))
    end

    L = s[end]
    # Uniform target arc-length positions
    sr = range(zero(T), L, length = Nout)

    # Output containers
    φr = similar(φ, d, Nout)
    payloadr = _payload_alloc_like(payload, Nout)

    # --- helper: linear interpolation at target s ---
    # For each target sr[j], find segment i-1 -> i s.t. s[i-1] <= sr <= s[i]
    @inbounds for j in 1:Nout
        sj = sr[j]
        i = searchsortedfirst(s, sj)  # i in 1..N
        if i <= 1
            # clamp to first point
            @inbounds for k in 1:d
                φr[k,j] = φ[k,1]
            end
            _payload_set!(payloadr, payload, j, 1, one(T), zero(T))
        elseif i > N
            # clamp to last point
            @inbounds for k in 1:d
                φr[k,j] = φ[k,N]
            end
            _payload_set!(payloadr, payload, j, N, one(T), zero(T))
        else
            s0, s1 = s[i-1], s[i]
            t = (s1 - s0) ≤ epslen ? zero(T) : (sj - s0) / (s1 - s0)

            @inbounds for k in 1:d
                φr[k,j] = (one(T)-t)*φ[k,i-1] + t*φ[k,i]
            end
            _payload_set!(payloadr, payload, j, i, t, one(T)-t)
        end
    end

    return φr, payloadr
end

# ---- payload helpers (supports vector length N or matrix k×N) ----

# Allocate payload-like container with Nout points
function _payload_alloc_like(payload::AbstractMatrix{T}, Nout::Int) where {T}
    k, _ = size(payload)
    return similar(payload, T, k, Nout)
end

# Write interpolated payload at output index j.
# Convention: i is the RIGHT endpoint index (segment i-1 -> i).
function _payload_set!(out::AbstractMatrix{T}, payload::AbstractMatrix{T},
                      j::Int, i::Int, t::T, w0::T) where {T}
    k, _ = size(payload)
    if i == 1
        @inbounds for r in 1:k
            out[r,j] = payload[r,1]
        end
    else
        @inbounds for r in 1:k
            out[r,j] = w0*payload[r,i-1] + t*payload[r,i]
        end
    end
end


function reparametrization(u_new, v_new, α, M, ϕ_a, ϕ_b)
    # u_new, v_new: (N, d)
    N, d = size(u_new)
    @assert size(v_new) == (N, d)
    @assert N == M+1

    ϕ = (u_new .+ v_new) ./ 2          # (N, d)
    θ = (u_new .- v_new) ./ (2α)       # (N, d)

    # Convert to (d, N) for arc-length code
    φmat = permutedims(ϕ)              # (d, N)
    payload = permutedims(θ)           # (d, N)  (payload can be matrix k×N)

    # Reparametrize to N points (i.e. M+1)
    φr, θr = reparametrization_arc_length(φmat, payload, M+1)

    # Convert back to (N, d)
    ϕr = permutedims(φr)               # (N, d)
    θr = permutedims(θr)               # (N, d)

    θr[1, :] .= θ[1, :]
    θr[end, :] .= 0
    ϕr[1, :] .= ϕ_a
    ϕr[end, :] .= ϕ_b

    u = ϕr .+ α .* θr
    v = ϕr .- α .* θr
    return u, v
end

function build_lambda_and_src(u, v, α, Δs, params; sign::Int)

    # u, v are (N,d)
    N, d = size(u)
    @assert size(v) == (N, d)

    ϕ = (u .+ v) ./ 2
    θ = (u .- v) ./ (2α)

    # gradients at nodes
    gθ = similar(u)
    gϕ = similar(u)

    H  = zeros(eltype(u), N)

    @inbounds for i in 1:N
        ϕi = collect(@view ϕ[i,:])
        θi = collect(@view θ[i,:])

        gθ[i,:] .= grad_theta_H(ϕi, θi, params)
        gϕ[i,:] .= grad_phi_H(ϕi, θi, params)
        H[i] = H_vals(ϕi, θi, params)
    end

    # src at nodes
    src = α .* gϕ .+ sign .* gθ

    # derivative of φ at nodes
    dϕ = similar(u)

    @inbounds begin
        # forward difference
        dϕ[1,:] .= (ϕ[2,:] .- ϕ[1,:]) ./ Δs

        # central differences
        for i in 2:N-1
            dϕ[i,:] .= (ϕ[i+1,:] .- ϕ[i-1,:]) ./ (2Δs)
        end

        # backward difference
        dϕ[N,:] .= (ϕ[N,:] .- ϕ[N-1,:]) ./ Δs
    end

    # λ at nodes
    λ = zeros(eltype(u), N)

    @inbounds for i in 1:N
        norm2   = zero(eltype(u))
        dotterm = zero(eltype(u))

        for k in 1:d
            δ = dϕ[i,k]
            norm2   += δ*δ
            dotterm += gθ[i,k] * δ
        end

        disc = dotterm^2 - 4 * H[i] * norm2
        denom = 2 * max(norm2, 1e-14)

        λ[i] = (abs(dotterm) + sqrt(max(0.0, disc))) / denom
    end

    return λ, src
end


function update_u(u, v, α, Δs, Δτ0, ϕ_b, params)
    # u, v are (N,2). ϕ_b should be length-2 (or 1×2) endpoint in ϕ-space.
    N = size(u, 1)
    M = N - 1

    λ, f = build_lambda_and_src(u, v, α, Δs, params; sign = -1)

    λmax = maximum(λ)
    Δτ = Δτ0#min(Δτ0, 0.2 * Δs / max(λmax, 1e-12))

    u_new = similar(u)

    # boundary: u_new[end,:] = -v[end,:] + 2*ϕ_b
    u_new[end, :] .= -v[end, :] .+ 2 .* ϕ_b
    u_new[M, :] .= (u[M, :] ./ Δτ  .+ (λ[M]/Δs) .* u_new[M+1, :]  .+ f[M, :]) ./ (1/Δτ + λ[M]/Δs)
    @inbounds for i in M-1:-1:1
        u_new[i, :] .= (u[i, :]./ Δτ .+ (λ[i, :]/(2*Δs)) .* (-u_new[i+2, :] .+ 4*u_new[i+1, :]) .+ f[i, :]) ./ (1/Δτ + 3*λ[i]/(2*Δs))
    end

    return u_new, Δτ
end


function update_v(u_new, v, α, Δs, Δτ, ϕ_a, params)
    # u_new, v are (N,2). ϕ_a should be length-2 (or 1×2) endpoint in ϕ-space.
    N = size(v, 1)
    M = N - 1

    λ, g = build_lambda_and_src(u_new, v, α, Δs, params; sign = +1)

    v_new = similar(v)

    # boundary: v_new[1,:] = -u_new[1,:] + 2*ϕ_a
    v_new[1, :] .= -u_new[1, :] .+ 2 .* ϕ_a
    v_new[2, :] .= (v[2, :]./Δτ .+ (λ[2]/Δs) .* v_new[1, :] .+ g[2, :]) ./ (1/Δτ + λ[2]/Δs)
    @inbounds for i in 3:M+1
        λi = λ[i]
        v_new[i, :] .= (v[i, :] ./ Δτ .+ (λi/(2*Δs)) .* (4*v_new[i-1, :].-v_new[i-2, :]) .+ g[i, :]) ./ (1/Δτ +3*λi / (2*Δs))
    end

    return v_new
end

function compute_cost(u, v, α, Δs, params)

    ϕ = (u .+ v) ./ 2
    θ = (u .- v) ./ (2α)

    N, d = size(ϕ)

    # λ at nodes
    λ, _ = build_lambda_and_src(u, v, α, Δs, params; sign=-1)

    # Hamiltonian at nodes
    H = zeros(eltype(u), N)

    @inbounds for i in 1:N
        ϕi = collect(@view ϕ[i,:])
        θi = collect(@view θ[i,:])
        H[i] = H_vals(ϕi, θi, params)
    end

    cost = zero(eltype(u))

    @inbounds for i in 1:N-1

        # segment derivative
        dϕds = (ϕ[i+1,:] .- ϕ[i,:]) ./ Δs

        # midpoint θ
        θmid = (θ[i,:] .+ θ[i+1,:]) ./ 2

        # <φ',θ>
        dotval = zero(eltype(u))
        for k in 1:d
            dotval += dϕds[k] * θmid[k]
        end

        # trapezoidal H/λ
        Hλ_i   = H[i]   / max(λ[i],   1e-14)
        Hλ_ip1 = H[i+1] / max(λ[i+1], 1e-14)
        Hλ_mid = 0.5 * (Hλ_i + Hλ_ip1)

        cost += Δs * (dotval - Hλ_mid)
    end

    return cost
end

function residuals(u, v, α, M, params)
    Δs = 1 / M
    N, d = size(u)
    @assert size(v) == (N, d)
    @assert N == M + 1

    # λ and source terms
    λu, f = build_lambda_and_src(u, v, α, Δs, params; sign = -1)
    λv, g = build_lambda_and_src(u, v, α, Δs, params; sign = +1)

    Ru = zeros(eltype(u), N, d)
    Rv = zeros(eltype(v), N, d)

    # u-equation residuals
    Ru[M, :] .= λu[M] .* (u[M, :] .- u[M+1, :]) ./ Δs .- f[M, :]

    @inbounds for i in 1:M-1
        Ru[i, :] .= λu[i] .* (3u[i, :] .- 4u[i+1, :] .+ u[i+2, :]) ./ (2Δs) .- f[i, :]
    end

    # v-equation residuals
    Rv[2, :] .= λv[2] .* (v[2, :] .- v[1, :]) ./ Δs .- g[2, :]

    @inbounds for i in 3:N
        Rv[i, :] .= λv[i] .* (3v[i, :] .- 4v[i-1, :] .+ v[i-2, :]) ./ (2Δs) .- g[i, :]
    end

    # nodewise residual magnitude
    res_nodewise = zeros(eltype(u), N)

    @inbounds for i in 2:M
        res_nodewise[i] = sum(abs.(Ru[i, :])) + sum(abs.(Rv[i, :]))
    end

    res_max = maximum(res_nodewise[2:M])

    return res_max
end
    

In [ ]:
const ∞ = Inf
ħ=1
e=1
β = ∞ # Case zero temperature T=0, β → ∞
ϕ_quantum=1/2
#### Caracterization of the circuit #####
ω_a = 25.338776456203686
ω_b = 2*ω_a

φ_a = 0.11
Z_a = 2*ϕ_quantum^2*φ_a^2
C_a_inv = Z_a*ω_a
E_Ca = C_a_inv/2

φ_b = 0.204
Z_b = 2*ϕ_quantum^2*φ_a^2
C_b_inv = Z_b*ω_b
E_Cb = C_b_inv/2

E_J = 37.12*2pi /5

#Environment
κ_b=5/10.4; # 1/ns from Leon's paper

ω_d=ω_b # Drive frequency
ω_p=2*ω_a-ω_d # Pump Frequency

ε_p = 0.1
g2 = besselj(1,ε_p)*E_J*φ_a^2*φ_b # (Leon's paper)
ε_d = 10.12*g2#0.75*g2_max

β_a_p, β_b_p = 2*E_J*sin(ε_p)*φ_a^2/(ω_a), 2*E_J*sin(ε_p)*φ_b^2/(ω_b)
ε_d_p = ε_d*φ_b/ω_b
κ_b_p = κ_b / ω_b

ℵ = β_a_p*β_b_p/(4*κ_b_p)
ℶ = β_a_p*ε_d_p/(2*κ_b_p)

In [ ]:
function equation_2modes!(du,u,params,t)
    β_a_p,β_b_p,ε_d_p,κ_b_p = params

    p_a = u[1]
    p_b = u[2]
        
    du[1] = -β_a_p*p_a*p_b/sqrt(2)
    du[2] = 2*(β_b_p*p_a^2/2/sqrt(2)-ε_d_p/sqrt(2)-κ_b_p*p_b)
    return nothing
end
t_min = 0.0
t_max = 1e3/κ_b_p 
tspan = (t_min, t_max)
M = 200

a_0 = 0.0-0.01im 
b_0 = -1im*ε_d_p/2/κ_b_p#0

p_a0, p_b0 = imag(a_0)*sqrt(2), imag(b_0)*sqrt(2)
u_0 = [p_a0, p_b0]
params = (β_a_p,β_b_p,ε_d_p,κ_b_p)
prob = ODEProblem(equation_2modes!, u_0, tspan, params, saveat = range(t_min, t_max, length = M+1))
sol = solve(prob, Tsit5())
t = sol.t
p_a = [u[1] for u in sol.u]
p_b = [u[2] for u in sol.u];
x_all = -.2:0.01:.2
p_all = -.35:0.01:.35

U = [ℵ*(x^2 + p^2)^2/4 + ℶ*(x^2 - p^2)/2 for p in p_all, x in x_all]

heatmap(x_all, p_all, U)
plot!(zeros(length(p_a)),p_a, label="Trajectory")

In [ ]:
plot(p_a, p_b)
xlabel!("p_a")
ylabel!("p_b")

In [ ]:
# ---------------- basic physics functions ----------------
function grad_phi_H(ϕ::AbstractVector{<:Real}, θ::AbstractVector{<:Real}, params)
    p_a, p_b = ϕ
    P_p_a, P_p_b = θ
    ϵ,β_a_p,β_b_p,ε_d_p,κ_b_p = params
    grad =@. [-β_a_p*(p_b*P_p_a)/sqrt(2)+2β_b_p*(p_a*P_p_b)/sqrt(2),
            -β_a_p*(p_a*P_p_a)/sqrt(2)-2κ_b_p*P_p_b]
    return(grad)
end

function grad_theta_H(ϕ::AbstractVector{<:Real}, θ::AbstractVector{<:Real}, params)
    p_a, p_b = ϕ
    P_p_a, P_p_b = θ
    ϵ,β_a_p,β_b_p,ε_d_p,κ_b_p = params
    grad =@. [-β_a_p*(p_a*p_b)/sqrt(2),
            2β_b_p*p_a^2/2/sqrt(2)-2ε_d_p/sqrt(2)-2κ_b_p*p_b+2*ϵ*P_p_b
            ]
    return(grad)
end

function H_vals(ϕ::AbstractVector{<:Real}, θ::AbstractVector{<:Real}, params)
    p_a, p_b = ϕ
    P_p_a, P_p_b = θ
    ϵ,β_a_p,β_b_p,ε_d_p,κ_b_p = params
    return @.(-β_a_p*(p_a*p_b)/sqrt(2))*P_p_a +
            2(-β_b_p*(-p_a^2)/2/sqrt(2)-ε_d_p/sqrt(2)-κ_b_p*p_b)*P_p_b +
            ϵ*(P_p_b^2)
end

In [ ]:
ϵ = 1e-3
ϕ_a = [-sqrt(2*ε_d_p/β_b_p),0.0]
ϕ_b = [0.0, -sqrt(0.5)*ε_d_p/κ_b_p]
params = (ϵ,β_a_p,β_b_p,ε_d_p,κ_b_p)

ϕ_diss = cat(p_a[end:-1:1], p_b[end:-1:1], dims = 2)
ϕ_init = cat(collect(range(ϕ_a[1],ϕ_b[1], length = M+1)), collect(range(ϕ_a[2],ϕ_b[2], length = M+1)), dims = 2);

In [ ]:
α = 0.01
θ_init = zeros(M+1,length(ϕ_a))
θ_init[1,2]=-1e-1

u = ϕ_init .+ α .* θ_init
v = ϕ_init .- α .* θ_init

Δs = 1 / (M)
Δτ0 = 1e-1
N_iter = 1e4
N_compute_action = 100
idx_compute_action = div(N_iter,N_compute_action)
S_list = []
max_res_list = []

for n in 1:N_iter
    u_new, Δτ = update_u(u, v, α, Δs, Δτ0, ϕ_b, params)
    v_new     = update_v(u_new, v, α, Δs, Δτ, ϕ_a, params)
    u, v = reparametrization(u_new, v_new, α, M, ϕ_a, ϕ_b)
    if mod(n,idx_compute_action)==0
        S = compute_cost(u, v, α, Δs, params)
        append!(S_list, S)
        ϕ_chk = (u .+ v) ./ 2
        θ_chk = (u .- v) ./ (2α)
        
        Hchk = [H_vals(ϕ_chk[i, :], θ_chk[i, :], params) for i in 1:(M+1)]
        println("iter=$n, max|H|=", maximum(abs.(Hchk)))
        append!(max_res_list,residuals(u, v, α, M, params))
    end
end
ϕ = (u .+ v) ./ 2
θ = (u .- v) ./ (2α);
p_a = ϕ[:,1]
p_b = ϕ[:,2]
P_p_a = θ[:,1]
P_p_b = θ[:,2];

In [ ]:
plot(1:idx_compute_action:N_iter,S_list, ylabel="Action", color=:blue, label="")
xlabel!("Optimization step")
plot!(twinx(),(1:idx_compute_action:N_iter)[2:end], abs.(diff(S_list)), ylabel="diff Action", color=:red, yscale=:log10,label="")


In [ ]:
# --- subplot 1 ---
p1 = plot(P_p_a, p_a,
    xlabel=L"$P_{p_a}$",
    ylabel=L"$p_a$",
     label="GDA")

# --- subplot 2 ---
p2 = plot(P_p_b, p_b,
    xlabel=L"$P_{p_b}$",
    ylabel=L"$p_b$",
    label="GDA")

# --- combine ---
plot(p1, p2, layout=(1,2))

In [ ]:
plot(p_a, p_b)
xlabel!("p_a")
ylabel!("p_b")

In [ ]:
H_fin_array = [H_vals(ϕ[i,:], θ[i,:], params) for i in 1:size(ϕ,1)]
plot(1:(M+1), H_fin_array)
xlabel!("Path")
ylabel!("Hamiltonian")

In [ ]:
plot(P_p_b, H_fin_array)

In [ ]:
plot(1:idx_compute_action:N_iter,max_res_list)
xlabel!("Optimization step")
ylabel!("Max Sum Residual")

In [ ]:
E_J = 37.12*2pi

#Environment
κ_b=1/10.4; # 1/ns from Leon's paper

ω_d=ω_b # Drive frequency
ω_p=2*ω_a-ω_d # Pump Frequency

ε_p = 0.2
g2 = besselj(1,ε_p)*E_J*φ_a^2*φ_b # (Leon's paper)
ε_d = 10.12*g2#0.75*g2_max

β_a_p, β_b_p = 2*E_J*sin(ε_p)*φ_a^2/(ω_a), 2*E_J*sin(ε_p)*φ_b^2/(ω_b)
ε_d_p = ε_d*φ_b/ω_b
κ_b_p = κ_b / ω_b

ℵ = β_a_p*β_b_p/(4*κ_b_p)
ℶ = β_a_p*ε_d_p/(2*κ_b_p)
t_min = 0.0
t_max = 1e1/κ_b_p 
tspan = (t_min, t_max)
M = 1000

a_0 = 0.0-0.01im 
b_0 = -1im*ε_d_p/2/κ_b_p#0

p_a0, p_b0 = imag(a_0)*sqrt(2), imag(b_0)*sqrt(2)
u_0 = [p_a0, p_b0]
params = (β_a_p,β_b_p,ε_d_p,κ_b_p)
prob = ODEProblem(equation_2modes!, u_0, tspan, params, saveat = range(t_min, t_max, length = M+1))
sol = solve(prob, Tsit5())
t = sol.t
p_a = [u[1] for u in sol.u]
p_b = [u[2] for u in sol.u];
x_all = -.2:0.01:.2
p_all = -.35:0.01:.35

U = [ℵ*(x^2 + p^2)^2/4 + ℶ*(x^2 - p^2)/2 for p in p_all, x in x_all]

heatmap(x_all, p_all, U)
plot!(zeros(length(p_a)),p_a, label="Trajectory")

In [ ]:
ϵ = 1e-3
ϕ_a = [-sqrt(2*ε_d_p/β_b_p), 0.0]
ϕ_b = [0.0, -sqrt(0.5)*ε_d_p/κ_b_p]
params = (ϵ,β_a_p,β_b_p,ε_d_p,κ_b_p)
ϕ_diss = cat(p_a[end:-1:1], p_b[end:-1:1], dims = 2)
ϕ_init = cat(collect(range(ϕ_a[1],ϕ_b[1], length = M+1)), collect(range(ϕ_a[2],ϕ_b[2], length = M+1)), dims = 2);#permutedims(reparametrize_φ_alone(permutedims(ϕ_diss)))#
plot(ϕ_init)

In [ ]:
ϵ = 1e-3
ϕ_a = [-sqrt(2*ε_d_p/β_b_p), 0.0]
ϕ_b = [0.0, -sqrt(0.5)*ε_d_p/κ_b_p]
params = (ϵ,β_a_p,β_b_p,ε_d_p,κ_b_p)
α = 0.1

θ_init = zeros(M+1,length(ϕ_a))/1e2
u = ϕ_init .+ α .* θ_init
v = ϕ_init .- α .* θ_init

Δs = 1 / (M+1)
Δτ0 = 1e-3
N_iter = 4e4
N_compute_action = 200
idx_compute_action = div(N_iter,N_compute_action)
S_list = []
max_res_list = []

for n in 1:N_iter
    u_new, Δτ = update_u(u, v, α, Δs, Δτ0, ϕ_b, params)
    v_new     = update_v(u_new, v, α, Δs, Δτ, ϕ_a, params)
    u, v      = reparametrization(u_new, v_new, α, M, ϕ_a, ϕ_b)

    if mod(n,idx_compute_action)==0
        S = compute_cost(u, v, α, Δs, params)
        append!(S_list, S)
        ϕ_chk = (u .+ v) ./ 2
        θ_chk = (u .- v) ./ (2α)
        
        Hchk = [H_vals(ϕ_chk[i, :], θ_chk[i, :], params) for i in 1:(M+1)]
        println("iter=$n, max|H|=", maximum(abs.(Hchk)))
        append!(max_res_list,residuals(u, v, α, M, params))
    end
end
ϕ = (u .+ v) ./ 2
θ = (u .- v) ./ (2α);
p_a = ϕ[:,1]
p_b = ϕ[:,2]
P_p_a = θ[:,1]
P_p_b = θ[:,2];

In [ ]:
plot(1:idx_compute_action:N_iter,S_list, ylabel="Action", color=:blue, label="")
xlabel!("Optimization step")
plot!(twinx(),(1:idx_compute_action:N_iter)[2:end], abs.(diff(S_list)), ylabel="diff Action", color=:red, yscale=:log10,label="")


In [ ]:

# --- subplot 1 ---
p1 = plot(P_p_a, p_a,
    xlabel=L"$P_{p_a}$",
    ylabel=L"$p_a$",
     label="GDA")

# --- subplot 2 ---
p2 = plot(P_p_b, p_b,
    xlabel=L"$P_{p_b}$",
    ylabel=L"$p_b$",
    label="GDA")

# --- combine ---
plot(p1, p2, layout=(1,2))

In [ ]:
H_fin_array = [H_vals(ϕ[i,:], θ[i,:], params) for i in 1:size(ϕ,1)]
plot(1:(M+1), H_fin_array)
xlabel!("Path")
ylabel!("Hamiltonian")

In [ ]:
plot(1:idx_compute_action:N_iter,max_res_list)
xlabel!("Optimization step")
ylabel!("Mas Sum Residual")